In [30]:
# %pip install requests pandas numpy scikit-learn bertopic sentence-transformers \
#              vaderSentiment matplotlib seaborn wordcloud plotly tqdm

In [21]:
import requests
import time
import datetime as dt

HEADERS = {"User-Agent": "fashion_trend_poc:v0.1 (academic research project)"}
BASE_URL = "https://www.reddit.com"
REQUEST_DELAY = 1.5  # seconds between requests


def fetch_posts(subreddit: str, sort: str = "new", limit: int = 500) -> list:
    posts, after = [], None

    while len(posts) < limit:
        params = {"limit": 100, "raw_json": 1}
        if after:
            params["after"] = after

        url = f"{BASE_URL}/r/{subreddit}/{sort}.json"
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=30)
        except requests.exceptions.ReadTimeout:
            print(f"  Timeout fetching posts page for r/{subreddit} — stopping pagination")
            break

        if response.status_code == 429:
            print("  Rate limited — waiting 60s...")
            time.sleep(60)
            continue
        if response.status_code != 200:
            print(f"  Error {response.status_code} on r/{subreddit} — skipping")
            break

        data = response.json()["data"]
        batch = data["children"]
        if not batch:
            break

        posts.extend(batch)
        after = data.get("after")
        if not after:
            break

        time.sleep(REQUEST_DELAY)

    return posts[:limit]


def fetch_comments(post_id: str, subreddit: str, top_n: int = 5) -> list:
    url = f"{BASE_URL}/r/{subreddit}/comments/{post_id}.json"
    try:
        response = requests.get(
            url, headers=HEADERS, params={"raw_json": 1, "limit": top_n}, timeout=30
        )
    except requests.exceptions.ReadTimeout:
        return []  # skip this post's comments silently

    if response.status_code != 200:
        return []

    try:
        comment_listing = response.json()[1]["data"]["children"]
        return [
            c["data"] for c in comment_listing
            if c["kind"] == "t1"
            and c["data"].get("body") not in (None, "[deleted]", "[removed]")
        ][:top_n]
    except (IndexError, KeyError):
        return []


def ts_to_dt(timestamp):
    """Convert Unix timestamp to timezone-aware UTC datetime."""
    return dt.datetime.fromtimestamp(timestamp, tz=dt.timezone.utc)


print("Client ready.")

Client ready.


In [22]:
SUBREDDITS = [
    "femalefashionadvice",
    "malefashionadvice",
    "streetwear",
    "fashion",
    "rawdenim",
    "frugalmalefashion",
    "weddingplanning",
]

POSTS_PER_SUB    = 50
SORT_BY          = "new"
INCLUDE_COMMENTS = True
TOP_N_COMMENTS   = 10

In [23]:
import pandas as pd
import os
from tqdm import tqdm

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

all_records = []

for sub_name in tqdm(SUBREDDITS, desc="Subreddits"):
    checkpoint_path = f"{CHECKPOINT_DIR}/{sub_name}.csv"

    if os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        all_records.append(existing)
        print(f"  Loaded r/{sub_name} from checkpoint ({len(existing):,} records)")
        continue

    print(f"\nFetching r/{sub_name}...")
    posts = fetch_posts(sub_name, sort=SORT_BY, limit=POSTS_PER_SUB)
    print(f"  Got {len(posts)} posts")

    records = []
    timeouts = 0

    for raw in tqdm(posts, desc=sub_name, leave=False):
        p = raw["data"]
        created_utc = ts_to_dt(p["created_utc"])

        records.append({
            "id":           p["id"],
            "subreddit":    sub_name,
            "type":         "post",
            "author":       p.get("author", ""),
            "title":        p.get("title", ""),
            "text":         p.get("selftext", ""),
            "full_text":    (p.get("title", "") + " " + p.get("selftext", "")).strip(),
            "score":        p.get("score", 0),
            "num_comments": p.get("num_comments", 0),
            "upvote_ratio": p.get("upvote_ratio", None),
            "url":          f"https://reddit.com{p.get('permalink', '')}",
            "author_flair": p.get("author_flair_text", None),
            "post_flair":   p.get("link_flair_text", None),
            "created_utc":  created_utc,
            "week":         created_utc.strftime("%Y-W%V"),
            "month":        created_utc.strftime("%Y-%m"),
        })

        if INCLUDE_COMMENTS:
            comments = fetch_comments(p["id"], sub_name, top_n=TOP_N_COMMENTS)
            if not comments:
                timeouts += 1
            for c in (comments or []):
                c_created = ts_to_dt(c["created_utc"])
                records.append({
                    "id":           c["id"],
                    "subreddit":    sub_name,
                    "type":         "comment",
                    "author":       c.get("author", ""),
                    "title":        p.get("title", ""),
                    "text":         c.get("body", ""),
                    "full_text":    c.get("body", ""),
                    "score":        c.get("score", 0),
                    "num_comments": None,
                    "upvote_ratio": None,
                    "url":          f"https://reddit.com{c.get('permalink', '')}",
                    "author_flair": c.get("author_flair_text", None),
                    "post_flair":   p.get("link_flair_text", None),
                    "created_utc":  c_created,
                    "week":         c_created.strftime("%Y-W%V"),
                    "month":        c_created.strftime("%Y-%m"),
                })
            time.sleep(REQUEST_DELAY)

    df_sub = pd.DataFrame(records)
    df_sub.to_csv(checkpoint_path, index=False)
    all_records.append(df_sub)
    print(f"  Saved checkpoint: {checkpoint_path}  ({len(df_sub):,} records, {timeouts} posts with no comments)")

Subreddits:   0%|          | 0/7 [00:00<?, ?it/s]

  Loaded r/femalefashionadvice from checkpoint (21 records)
  Loaded r/malefashionadvice from checkpoint (217 records)
  Loaded r/streetwear from checkpoint (273 records)

Fetching r/fashion...
  Got 50 posts


Subreddits:  57%|█████▋    | 4/7 [01:31<01:08, 22.78s/it]

  Saved checkpoint: checkpoints/fashion.csv  (285 records, 3 posts with no comments)

Fetching r/rawdenim...
  Got 50 posts


Subreddits:  71%|███████▏  | 5/7 [03:03<01:22, 41.08s/it]

  Saved checkpoint: checkpoints/rawdenim.csv  (262 records, 7 posts with no comments)

Fetching r/frugalmalefashion...
  Rate limited — waiting 60s...
  Rate limited — waiting 60s...
  Rate limited — waiting 60s...
  Got 50 posts


Subreddits:  86%|████████▌ | 6/7 [07:35<01:44, 104.24s/it]

  Saved checkpoint: checkpoints/frugalmalefashion.csv  (305 records, 0 posts with no comments)

Fetching r/weddingplanning...
  Got 50 posts


Subreddits: 100%|██████████| 7/7 [09:06<00:00, 78.08s/it] 

  Saved checkpoint: checkpoints/weddingplanning.csv  (232 records, 8 posts with no comments)


In [25]:
import re
import glob
import pandas as pd

# ── Combine all checkpoint CSVs ───────────────────────────────────────────────
csv_files = glob.glob("checkpoints/*.csv")
if not csv_files:
    raise FileNotFoundError("No checkpoint files found. Run the scraping cell first.")

df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Loaded {len(df_raw):,} raw records from {len(csv_files)} subreddit(s)")

# ── Drop image-only posts ─────────────────────────────────────────────────────
# Image posts have an empty selftext body — only their title came through.
# Comments always have text, so we keep all of those.
IMAGE_HOSTS = ("i.redd.it", "i.imgur.com", "imgur.com", "preview.redd.it", "gallery")

def is_image_post(row) -> bool:
    if row["type"] != "post":
        return False
    text = str(row.get("text", "")).strip()
    url  = str(row.get("url", "")).lower()
    return text == "" and any(host in url for host in IMAGE_HOSTS)

before = len(df_raw)
df_raw = df_raw[~df_raw.apply(is_image_post, axis=1)].reset_index(drop=True)
print(f"Dropped {before - len(df_raw):,} image-only posts → {len(df_raw):,} records remaining")
print(df_raw["type"].value_counts().to_string())

# ── Clean text ────────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", "", text)        # strip URLs
    text = re.sub(r"r/\w+|u/\w+", "", text)   # strip subreddit/user refs
    text = re.sub(r"[^\w\s']", " ", text)      # keep words + apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

df = df_raw.copy()
df["clean_text"] = df["full_text"].apply(clean_text)

# Drop anything still empty or duplicate after cleaning
df = df[
    (df["clean_text"].str.len() > 20) &
    (~df["clean_text"].isin(["deleted", "removed"]))
].drop_duplicates(subset="clean_text").reset_index(drop=True)

print(f"\n{len(df):,} documents ready for analysis")
df[["type", "subreddit", "clean_text"]].head(5)

Loaded 1,595 raw records from 7 subreddit(s)
Dropped 0 image-only posts → 1,595 records remaining
type
comment    1290
post        305

1,346 documents ready for analysis


,type,subreddit,clean_text
0,post,frugalmalefashion,25 back at new balance today only 4 23 2026 if...
1,comment,frugalmalefashion,i got 30 but make sure you check exclusions
2,comment,frugalmalefashion,make sure you check exclusions oh all the good...
3,comment,frugalmalefashion,not eligible on any 990 shoes that's dumb
4,comment,frugalmalefashion,man zappos had 20 off 990s a few days ago i sh...


In [29]:
df.to_parquet("reddit_outputs.parquet")